# CleanModel — WavLM-large Fine-tuning (reads from SQLCLEAN's local SQLite db)

**Task:** 6-class SER — Angry · Disgusted · Fearful · Happy · Neutral · Sad
**Data:** `Notebooks/ser_local.db` → `audio_dataset_w2v2`, built by **`SQLCLEAN.ipynb`** (run that notebook first)
**Why a separate notebook from `wav2vec2f.ipynb`:** that notebook re-downloads the raw dataset and
re-implements filtering + speaker split every run. Here, all of that "which clips make the cut" logic
already happened once in SQLCLEAN — this notebook only trains.

**Checkpoint reuse:** before extracting any audio or training anything, this notebook checks whether
`clean_ser_best/` already exists (e.g. unzipped from a Colab/Kaggle run) — if so, it loads that model
and skips straight to evaluation. If not found, it extracts the audio, trains from scratch, and saves
the result to `clean_ser_best/`. Either way it always works: reuse when there's something to reuse,
train fresh when there isn't.

1. **Environment** — local-only: imports, device check, package guard
2. **Data loading** — three `SELECT ... WHERE split = ?` queries against `ser_local.db`, label encoding, checkpoint check
3. **Audio pipeline** — load the already-16kHz clips (train set skipped if reusing a checkpoint), light augmentation, batch collation
4. **Model architecture** — WavLM-large (300 M), frozen conv encoder, trainable transformer + head
5. **Training** — LLRD optimizer, focal loss, cosine schedule, early stopping on val F1
6. **Loss curves** — train/val loss per epoch, val accuracy and F1-macro per epoch
7. **Evaluation** — test accuracy, per-class F1, confusion matrix (counts + normalised), CREMA-D vs RAVDESS breakdown
8. **Save & analysis** — export model, what worked, what didn't, what's next

> Same seed / split params as `SQLCLEAN.ipynb` (and therefore the same speaker-disjoint test set as
> `wav2vec2f.ipynb`), so accuracy stays directly comparable — this notebook only changes *where the
> data selection happens*, not the selection itself.


---
## 1 · Environment & constants

Local run only — this notebook reads `Notebooks/ser_local.db` and `Emotions_w2v2_16k/` from disk, so
it doesn't run on Kaggle/Colab like `wav2vec2f.ipynb` does. No dataset download, no Drive mount.


In [ ]:
# === Setup / imports (local only -- no Kaggle/Colab detection, no dataset download) ===
import sys, subprocess, os, json
import torch
from pathlib import Path
from packaging.version import parse

OUT_DIR = Path(".")
DEVICE  = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("Device      :", DEVICE)

N_GPU = torch.cuda.device_count()
if N_GPU:
    print("GPUs        :", N_GPU, [torch.cuda.get_device_name(i) for i in range(N_GPU)])

# --- Package check ---
need = False
try:
    import transformers, accelerate
    if parse(transformers.__version__) < parse("4.41") or parse(accelerate.__version__) < parse("0.30"):
        need = True
except ImportError:
    need = True

if need:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers>=4.41", "accelerate>=0.30",
                    f"torch=={torch.__version__}"], check=True)
    print(">> installed/upgraded -- restart the kernel and run again")
else:
    import transformers
    print("transformers:", transformers.__version__, "-- ok")

import numpy as np
import pandas as pd


---
## 2 · Data loading

`SQLCLEAN.ipynb` already did the file-level filtering, the 1–5 s duration cut, the CREMA-D + RAVDESS
scoping, the 'Suprised' drop, and the speaker-independent 3-way split. This notebook just queries the
result — no filtering, no splitting, no dataset download.


In [ ]:
# === Read the pre-selected, pre-split dataset straight from ser_local.db ===
from sqlalchemy import create_engine, text

DB_PATH = Path("ser_local.db")
if not DB_PATH.exists():
    raise FileNotFoundError(
        f"{DB_PATH} not found -- run SQLCLEAN.ipynb first to build the local dataset.")

engine = create_engine(f"sqlite:///{DB_PATH}")

def load_split(split):
    q = text("SELECT clip_id, filename, processed_path AS path, emotion, source, speaker, split "
              "FROM audio_dataset_w2v2 WHERE split = :split")
    with engine.connect() as conn:
        return pd.read_sql(q, conn, params={"split": split})

dftrain = load_split("train")
dfval   = load_split("val")
dftest  = load_split("test")
print(f"train {len(dftrain)} / val {len(dfval)} / test {len(dftest)} clips  (selected + split by SQLCLEAN)")
print(f"speakers -- train {dftrain['speaker'].nunique()}, "
      f"val {dfval['speaker'].nunique()}, test {dftest['speaker'].nunique()}")

# sanity check only -- SQLCLEAN already guarantees this, we just don't trust silently
for a, b, nm in [(dftrain, dfval, "train/val"), (dftrain, dftest, "train/test"), (dfval, dftest, "val/test")]:
    assert not (set(a["speaker"]) & set(b["speaker"])), f"speaker leakage in {nm}"
print("no speaker leakage across splits (re-verified here)")


### 2.1 · Labels

`LabelEncoder` fit on train only — same mapping reused for val and test.


In [ ]:
# === Labels ===
from sklearn.preprocessing import LabelEncoder
le   = LabelEncoder()
ytr  = le.fit_transform(dftrain["emotion"])
yval = le.transform(dfval["emotion"])
yte  = le.transform(dftest["emotion"])
CLASSES = list(le.classes_)
NUM = len(CLASSES)
print("classes:", CLASSES)


In [ ]:
# === Checkpoint check -- decide BEFORE extracting any audio whether we need to train at all ===
# If clean_ser_best/ already exists (e.g. unzipped here from a Colab/Kaggle run's clean_ser_best.zip),
# reuse it and skip straight to evaluation. Otherwise train from scratch, same as always.
SAVE_PATH = Path("clean_ser_best")

SKIP_TRAINING = SAVE_PATH.exists()
if SKIP_TRAINING:
    missing = [f for f in ("config.json", "model.safetensors", "preprocessor_config.json")
               if not (SAVE_PATH / f).exists()]
    if missing:
        print(f"{SAVE_PATH} exists but is missing {missing} -- treating as not found, will train fresh.")
        SKIP_TRAINING = False
    else:
        print(f"Found {SAVE_PATH} -- loading it, skipping feature extraction + training.")
else:
    print(f"No {SAVE_PATH} found -- will extract features and train from scratch.")


---
## 3 · Audio pipeline & dataset

Clips are already 16 kHz and capped at 5 s (SQLCLEAN did the resample) — this just loads them into
RAM. If a checkpoint was found above, the (expensive) training-set load is skipped entirely — only
val/test are loaded, since those are needed for evaluation either way. Train set gets light
augmentation when it is loaded:

| Augmentation | Prob | Detail |
|-------------|------|--------|
| Gaussian noise | 50 % | σ = 0.003 |
| Time-stretch | 30 % | rate 0.9–1.1 |

The `Collator` pads batches and applies WavLM's expected normalisation.


In [ ]:
# === Audio pipeline: load the pre-resampled 16 kHz clips into RAM ===
from transformers import AutoFeatureExtractor
import librosa
from tqdm.auto import tqdm

MODEL_ID = "microsoft/wavlm-large"
fe  = AutoFeatureExtractor.from_pretrained(MODEL_ID)
SR      = fe.sampling_rate          # 16000
MAX_LEN = int(SR * 5.0)
print(f"Model  : {MODEL_ID}")
print(f"SR     : {SR}  |  MAX_LEN: {MAX_LEN} samples ({MAX_LEN/SR:.0f}s)")

class WavDataset(torch.utils.data.Dataset):
    def __init__(self, frame, labels, desc, augment=False):
        self.wavs = [librosa.load(p, sr=SR)[0][:MAX_LEN].astype("float32")
                     for p in tqdm(list(frame["path"]), desc=desc)]
        self.labels  = list(labels)
        self.augment = augment
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        wav = self.wavs[i].copy()
        if self.augment:
            if np.random.rand() < 0.5:
                wav += np.random.randn(len(wav)).astype("float32") * 0.003
            if np.random.rand() < 0.3:
                rate = np.random.uniform(0.9, 1.1)
                wav  = librosa.effects.time_stretch(wav, rate=rate)[:MAX_LEN]
                wav  = wav.astype("float32")
        return {"input_values": wav, "label": int(self.labels[i])}

# Val/test are always needed for evaluation. Train is only extracted if we're actually going to
# train -- skips loading ~6.5k clips into RAM when we're just going to load a saved model instead.
val_ds  = WavDataset(dfval,  yval, "load-val")
test_ds = WavDataset(dftest, yte,  "load-test")
if SKIP_TRAINING:
    train_ds = None
    print(f"val/test items: {len(val_ds)}, {len(test_ds)}  (train clips NOT extracted -- reusing {SAVE_PATH})")
else:
    train_ds = WavDataset(dftrain, ytr, "load-train", augment=True)
    print("train/val/test items:", len(train_ds), len(val_ds), len(test_ds))


In [ ]:
# === Collator: pad each batch with the feature extractor (also does WavLM normalization) ===
from dataclasses import dataclass

@dataclass
class Collator:
    fe: AutoFeatureExtractor
    def __call__(self, batch):
        wavs   = [b["input_values"] for b in batch]
        labels = torch.tensor([b["label"] for b in batch], dtype=torch.long)
        out = self.fe(wavs, sampling_rate=self.fe.sampling_rate, padding=True, return_tensors="pt")
        out["labels"] = labels
        return out

collator = Collator(fe)


---
## 4 · Model architecture

`WavLMForSequenceClassification` (`microsoft/wavlm-large`) — conv encoder frozen, everything else trained.

| Component | Choice |
|-----------|--------|
| Backbone | WavLM-large — 300 M params, 24 layers, hidden=1024 |
| Feature encoder | Frozen (CNN front-end) |
| Weighted layer sum | Trainable scalar per layer — lets the model pick which layers matter for emotion |
| Head | Linear(1024 → 6) |
| Class weights | Balanced — corrects for Disgusted / Neutral being underrepresented |
| Trainable params | ~311 M |


In [ ]:
# === Model: WavLM-large + classification head ===
from transformers import WavLMForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

if SKIP_TRAINING:
    print(f"Loading fine-tuned weights from {SAVE_PATH} -- skipping base WavLM-large init/download")
    model = WavLMForSequenceClassification.from_pretrained(
        str(SAVE_PATH),
        num_labels=NUM,
        label2id={c: i for i, c in enumerate(CLASSES)},
        id2label={i: c for i, c in enumerate(CLASSES)},
        use_weighted_layer_sum=True,
        ignore_mismatched_sizes=True,
    ).to(DEVICE)
else:
    model = WavLMForSequenceClassification.from_pretrained(
        MODEL_ID,
        num_labels=NUM,
        label2id={c: i for i, c in enumerate(CLASSES)},
        id2label={i: c for i, c in enumerate(CLASSES)},
        use_weighted_layer_sum=True,
    ).to(DEVICE)
model.freeze_feature_encoder()

cw      = compute_class_weight("balanced", classes=np.arange(NUM), y=ytr)
class_w = torch.tensor(cw, dtype=torch.float32)
print("class weights:", {CLASSES[i]: round(float(w), 3) for i, w in enumerate(cw)})
n_layers  = model.wavlm.config.num_hidden_layers
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Backbone layers : {n_layers}")
print(f"Trainable params: {trainable:,}")


---
## 5 · Training

`WeightedTrainer` uses LLRD (lower transformer layers get smaller LR) and focal loss. Best checkpoint
picked by val F1-macro — test set not touched until Section 7.

| Setting | Value |
|---------|-------|
| Batch size | 32 (16 per device × 2 grad accum) |
| Base LR | 3e-5 |
| LLRD decay | 0.95 per layer from top |
| Schedule | Cosine + 400 warmup steps |
| Max epochs | 20 (early stop patience=4 on val F1) |
| Loss | Focal γ=2 + balanced class weights |
| Precision | bf16 if an A100-class GPU is available, else fp16/fp32 |


In [ ]:
# === Trainer: LLRD optimizer + focal loss + cosine schedule ===
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, f1_score

EFFECTIVE_BATCH  = 32
PER_DEVICE_BATCH = 16
GRAD_ACCUM = max(1, EFFECTIVE_BATCH // (PER_DEVICE_BATCH * max(1, N_GPU)))
print(f"GPUs: {N_GPU} | per_device: {PER_DEVICE_BATCH} | grad_accum: {GRAD_ACCUM} | "
      f"effective batch: {PER_DEVICE_BATCH * max(1, N_GPU) * GRAD_ACCUM}")

use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
use_fp16 = torch.cuda.is_available() and not use_bf16
print(f"Precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'fp32'}")

# --- LLRD optimizer ---
def get_llrd_optimizer(model, base_lr=3e-5, decay=0.95, wd=0.01):
    no_decay = {"bias", "LayerNorm.weight"}
    params   = []
    head_params = [(n, p) for n, p in model.named_parameters()
                   if "wavlm" not in n and p.requires_grad]
    params.append({"params": [p for _, p in head_params],
                   "lr": base_lr, "weight_decay": wd})
    for i, layer in enumerate(reversed(model.wavlm.encoder.layers)):
        lr_i = base_lr * (decay ** i)
        nd   = [p for n, p in layer.named_parameters() if any(nd in n for nd in no_decay)]
        wd_p = [p for n, p in layer.named_parameters() if not any(nd in n for nd in no_decay)]
        params += [{"params": wd_p, "lr": lr_i, "weight_decay": wd},
                   {"params": nd,   "lr": lr_i, "weight_decay": 0.0}]
    return torch.optim.AdamW(params)

# --- Focal loss trainer ---
class WeightedTrainer(Trainer):
    def create_optimizer(self):
        self.optimizer = get_llrd_optimizer(self.model, base_lr=3e-5, decay=0.95)
        return self.optimizer

    def compute_loss(self, model, inputs, return_outputs=False, **kw):
        labels = inputs.pop("labels")
        out    = model(**inputs)
        ce     = torch.nn.functional.cross_entropy(
            out.logits, labels,
            weight=class_w.to(out.logits.device),
            reduction="none",
        )
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** 2 * ce).mean()
        return (loss, out) if return_outputs else loss

def compute_metrics(p):
    logits = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    preds  = logits.argmax(-1)
    return {"acc":      accuracy_score(p.label_ids, preds),
            "f1_macro": f1_score(p.label_ids, preds, average="macro")}

HIST_PATH = SAVE_PATH / "history.json"

# ── Reuse the checkpoint found earlier, or build the Trainer and actually train ──
# Nothing below this point runs if we're reusing a checkpoint: no TrainingArguments, no Trainer
# instantiation (which would otherwise reference train_ds, unavailable when SKIP_TRAINING), no
# trainer.train().
if SKIP_TRAINING:
    print(f"Skipping training -- using weights already loaded from {SAVE_PATH}")
    log_history = json.load(open(HIST_PATH)) if HIST_PATH.exists() else []
    print(f"history entries: {len(log_history)}"
          + ("" if log_history else "  (none saved alongside this checkpoint)"))
else:
    args = TrainingArguments(
        output_dir=str(OUT_DIR / "clean_ser_ckpts"),
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=3e-5,
        num_train_epochs=20,
        warmup_steps=400,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        logging_steps=50,
        fp16=use_fp16,
        bf16=use_bf16,
        dataloader_num_workers=4,
        remove_unused_columns=False,
        ddp_find_unused_parameters=False,
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator, compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
    )

    print("No usable checkpoint -- training from scratch")
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    trainer.train()
    log_history = trainer.state.log_history
    SAVE_PATH.mkdir(parents=True, exist_ok=True)
    trainer.save_model(str(SAVE_PATH))
    fe.save_pretrained(str(SAVE_PATH))
    json.dump(log_history, open(HIST_PATH, "w"))
    print(f"Saved model + history -> {SAVE_PATH}")


---
## 6 · Loss curves

Train loss and val loss per epoch, plus val accuracy and F1-macro. Dashed line = best checkpoint epoch
(selected by highest val F1-macro — test set not touched until Section 7).


In [ ]:
# ── Loss curves from trainer.state.log_history (or saved history JSON) ───────
import matplotlib.pyplot as plt

train_logs = [e for e in log_history if "loss" in e and "eval_loss" not in e]
eval_logs  = [e for e in log_history if "eval_loss" in e]

if not eval_logs:
    print("No training history available -- model was loaded from a saved checkpoint without history.")
else:
    train_df        = pd.DataFrame(train_logs)
    train_df["ep"]  = train_df["epoch"].apply(lambda x: int(np.ceil(x)))
    train_per_ep    = train_df.groupby("ep")["loss"].mean().reset_index()
    tr_epochs       = train_per_ep["ep"].values
    tr_losses       = train_per_ep["loss"].values

    val_epochs = np.array([e["epoch"] for e in eval_logs])
    val_losses = np.array([e["eval_loss"] for e in eval_logs])
    val_accs   = np.array([e["eval_acc"] for e in eval_logs])
    val_f1s    = np.array([e.get("eval_f1_macro", 0) for e in eval_logs])

    best_ep = int(val_epochs[val_f1s.argmax()])
    print(f"Best checkpoint epoch  : {best_ep}")
    print(f"Val acc at best epoch  : {val_accs[val_f1s.argmax()]:.4f}")
    print(f"Val F1-macro at best   : {val_f1s.max():.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(tr_epochs, tr_losses, label="train loss", color="#1565C0")
    axes[0].plot(val_epochs, val_losses, label="val loss",  color="#C62828")
    axes[0].axvline(best_ep, color="black", ls="--", lw=1, label=f"best ckpt (ep {best_ep})")
    axes[0].set_xlabel("epoch"); axes[0].set_ylabel("loss (focal + class weights)")
    axes[0].set_title("Train vs val loss")
    axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

    axes[1].plot(val_epochs, val_accs, color="#1565C0", label="val acc")
    ax2 = axes[1].twinx()
    ax2.plot(val_epochs, val_f1s, color="#C62828", ls="--", label="val F1-macro")
    axes[1].axvline(best_ep, color="black", ls="--", lw=1)
    axes[1].set_xlabel("epoch"); axes[1].set_ylabel("val accuracy", color="#1565C0")
    ax2.set_ylabel("val F1-macro", color="#C62828")
    axes[1].set_title("Val accuracy & F1-macro per epoch")
    axes[1].grid(alpha=0.3)
    lines1, labels1 = axes[1].get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    axes[1].legend(lines1 + lines2, labels1 + labels2, fontsize=9)

    fig.suptitle("WavLM-large training history (CleanModel)", y=1.03)
    plt.tight_layout()
    plt.show()


### 6.1 · What the curves reveal

- **Fast early learning** — val loss drops steeply in epochs 1–5 as the weighted-layer-sum and classification head lock onto broad emotion signals. The frozen conv encoder means the model doesn't re-learn speech features from scratch.
- **Focal loss floor** — training loss never reaches zero; focal loss upweights hard examples so easy clips contribute less to the gradient.
- **Best checkpoint on F1-macro, not accuracy** — selecting on F1-macro is more sensitive to minority-class improvement (Disgusted, Neutral) than raw accuracy.
- **Fast convergence** — WavLM converges in 10–15 epochs. Pretrained representations need very little adaptation; most learning happens in the head and upper transformer layers.


---
## 7 · Evaluation on the held-out test set

Test set (speaker-independent, chosen once by SQLCLEAN, never seen during training or checkpoint
selection) evaluated once. Inference runs at batch=4 with immediate CPU offload to avoid OOM on the
300 M-param model.


In [ ]:
# === Full 6-class evaluation on the speaker-independent TEST set ===
import gc
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, precision_recall_fscore_support)
import seaborn as sns
from torch.utils.data import DataLoader

# Free optimizer states + flush fragmented memory (only if trainer was used)
try:
    if hasattr(trainer, "optimizer") and trainer.optimizer is not None:
        del trainer.optimizer
        trainer.optimizer = None
except NameError:
    pass
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU free after cleanup: {(torch.cuda.mem_get_info()[0]/1024**3):.1f} GB")

# Manual inference -- bypass trainer.predict() which accumulates all logits on GPU.
def run_inference(dataset, batch_size=4):
    loader = DataLoader(dataset, batch_size=batch_size, collate_fn=collator,
                        num_workers=2, pin_memory=False)
    model.eval()
    all_logits = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="inference"):
            batch.pop("labels", None)
            inputs = {k: v.to(DEVICE) for k, v in batch.items()}
            logits = model(**inputs).logits.cpu().float()
            all_logits.append(logits)
            del inputs, logits
    return torch.cat(all_logits, dim=0).numpy()

test_logits = run_inference(test_ds)
pred = test_logits.argmax(-1)

val_logits = run_inference(val_ds)
val_pred   = val_logits.argmax(-1)
print(f"VAL  -- acc {accuracy_score(yval, val_pred):.4f}  f1_macro {f1_score(yval, val_pred, average='macro'):.4f}")
print(f"\n--- Test set (speaker-independent) ---")
print(f"Test acc  : {accuracy_score(yte, pred):.4f}")
print(f"F1 macro  : {f1_score(yte, pred, average='macro'):.4f}")
print()
print(classification_report(yte, pred, target_names=CLASSES, digits=3))

for avg in ("macro", "weighted"):
    p, r, f1, _ = precision_recall_fscore_support(yte, pred, average=avg)
    print(f"{avg:8s} avg -- precision {p:.3f}  recall {r:.3f}  F1 {f1:.3f}")

print("\n--- Per-corpus accuracy ---")
for src in ["CREMA-D", "RAVDESS"]:
    mask = dftest["source"].values == src
    acc  = (pred[mask] == yte[mask]).mean()
    print(f"  {src}: {acc:.4f}  ({mask.sum()} clips)")

# ── Confusion matrix: raw counts + row-normalised (recall) ───────────────────
ens_acc = accuracy_score(yte, pred)
cm      = confusion_matrix(yte, pred)
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0], cbar=False)
axes[0].set_title("Counts")
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", vmin=0, vmax=1,
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1], cbar=False)
axes[1].set_title("Row-normalised (recall)")
for ax in axes:
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
fig.suptitle(
    f"CleanModel (WavLM-large) -- test acc {ens_acc:.3f}\n"
    f"SR=16 kHz | 1-5 s | CREMA-D + RAVDESS | speaker-independent | seed=42 | selection: SQLCLEAN",
    y=1.06)
plt.tight_layout()
plt.show()


---
## 8 · Save model & written analysis

### Where the model succeeds

- **Raw waveform input removes the feature ceiling** — WavLM operates on 16 kHz samples directly; its 24 transformer layers capture breathiness, micro-pitch variation, and voice quality that mel spectrograms average out.
- **High-arousal classes improve most** — Angry and Happy benefit from WavLM's fine-grained energy and attack-rate representations.
- **LLRD works as intended** — lower transformer layers (phonetics, prosody) are updated at a much smaller LR than the head and upper layers (semantics).
- **Selection is now reproducible and inspectable** — every "which clips count" decision lives in `SQLCLEAN.ipynb` and is queryable with plain SQL, instead of being re-derived (and potentially drifting) inside the training notebook every run.

### Where it still struggles

- **Disgusted remains the structural bottleneck** — Angry and Disgusted share high-arousal, negative-valence acoustic profiles; F1 for Disgusted is the lowest class in every run.
- **Memory pressure** — 300 M params mean batch=16 on GPU for training and batch=4 for inference.
- **Eval every epoch is slow** — 20 training epochs each with a full val pass is the dominant training cost.

### What we would change with more time

1. **Unfreeze the conv encoder** with gradient checkpointing enabled.
2. **LoRA fine-tuning** — adapter layers keep ~1 M trainable params instead of 311 M.
3. **Ensemble with the CNN baseline** — different inductive biases, likely different error patterns.
4. **Speaker k-fold CV** — one 15% test split has high variance with a modest number of speakers.


In [ ]:
# === Confirm which weights this run actually used ===
status = "reused (no training this run)" if SKIP_TRAINING else "freshly trained + saved this run"
print(f"Model path : {SAVE_PATH.resolve()}  ({status})")
print(f"Model      : {MODEL_ID}")
print(f"Classes    : {CLASSES}")
